In [1]:
import mmap
import time
from tqdm.notebook import tqdm
import os
import struct
from ensembles import Config, Ensemble, EnsembleFormatError, EnsembleWriter
import gc

PATH = "200x.000"
batch_size = 4096

In [2]:
def scan_mmap(path):
    with open(path, "rb") as f:
        file_size = os.path.getsize(path)
        mm = mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_READ)

        current_offset = 0
        ens_indexes = []

        while current_offset < file_size:
            if current_offset + 4 > file_size:
                break  # not enough bytes left for a header
        
            header = mm[current_offset:current_offset + 4]
        
            if header[0] == 0x7f and header[1] == 0x7f:
                ens_size = header[2] + (header[3] << 8) + 2
                if 32 <= ens_size <= 4096 and current_offset + ens_size <= file_size:
                    ens_indexes.append(current_offset)
                    current_offset += ens_size
                    continue
        
            current_offset += 1

        mm.close()
        return ens_indexes

ens_indexes = scan_mmap(PATH)

In [6]:
def decode_file_read(path, indexes):
    file = open(path, "rb")
    # Reset to head of file
    file.seek(0, 0)

    # Get the length of an ensemble
    file.seek(ens_indexes[0] + 2, 0)
    numbytes = struct.unpack("<h", file.read(2))[0]
    
    # Decode the first ensemble to get config and structure data
    file.seek(ens_indexes[0], 0)
    ens_dat = file.read(numbytes)
    ens = Ensemble.from_bytes(ens_dat)

    if not Ensemble.config:
        raise EnsembleFormatError(
            "Configuration data missing from first ensemble")
    cfg = Ensemble.config

    # Needed for velocity shapes
    n_cells = Ensemble.config.n_cells

    # List for storing ensembles, add first ensemble
    batch = []
    batch.append(ens)

    #writer = EnsembleWriter(fname, batch_size, n_cells)

    # For all ensembles, create object and write batches to file
    for i in range(1, len(ens_indexes)):
        file.seek(ens_indexes[i], 0)
        ens_dat = file.read(numbytes)
        ens = Ensemble.from_bytes(ens_dat)
        batch.append(ens)

        if len(batch) == batch_size:
            #writer.write_batch(batch)
            # Clear batch list and free up memory
            batch.clear()
            gc.collect()

    # Write any leftover batches
    if batch:
        #writer.write_batch(batch)
        # Clear batch list and free up memory
        batch.clear()
        gc.collect()

    file.close()

In [7]:
def decode_mmap(path, ens_indexes):
    # Memory-mapped file
    file = open(path, "rb")
    mm = mmap.mmap(file.fileno(), 0, access=mmap.ACCESS_READ)

    numbytes = struct.unpack("<h", mm[ens_indexes[0] + 2:ens_indexes[0] + 4])[0]
    ens_dat = mm[ens_indexes[0]:ens_indexes[0] + numbytes]
    ens = Ensemble.from_bytes(ens_dat)

    if not Ensemble.config:
        raise EnsembleFormatError("Configuration data missing from first ensemble")
    cfg = Ensemble.config

    batch = [ens]

    # For all ensembles, create object and write batches to file
    for i in range(1, len(ens_indexes)):
        ens_dat = mm[ens_indexes[i]:ens_indexes[i] + numbytes]
        ens = Ensemble.from_bytes(ens_dat)
        batch.append(ens)

        if len(batch) == batch_size:
            batch.clear()
            gc.collect()

    # Write any leftover batches
    if batch:
        batch.clear()
        gc.collect()

    mm.close()
    file.close()


In [8]:
%timeit decode_file_read(PATH, ens_indexes)

2232


KeyboardInterrupt: 

In [9]:
%timeit decode_mmap(PATH, ens_indexes)

2232


KeyboardInterrupt: 